In [0]:
from pyspark.sql.functions import (
    col, array_contains, size, lower, when, concat_ws, 
    current_timestamp, expr, lit, trim
)

# CATALOG & SCHEMA CONFIGURATION
CATALOG = "pipeline_signal"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA = "gold"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{GOLD_SCHEMA}")

def save_to_gold(df, table_name):
    full_table = f"{CATALOG}.{GOLD_SCHEMA}.{table_name}"
    df.write \
      .format("delta") \
      .mode("overwrite") \
      .option("overwriteSchema", "true") \
      .option("delta.enableChangeDataFeed", "true") \
      .saveAsTable(full_table)
    print(f"✅ Created Gold Table: `{full_table}` (Total Rows: {df.count()})")


# ---------------------------------------------------------
# 1. GOLD: Pipeline Impact & Incident Risk Mapping
# ---------------------------------------------------------
print("--- [1/2] Building Gold: Pipeline Impact & Risk Table ---")

df_silver_dbt = spark.read.table(f"{CATALOG}.{SILVER_SCHEMA}.silver_dbt_models")
df_silver_issues = spark.read.table(f"{CATALOG}.{SILVER_SCHEMA}.silver_github_issues")
df_silver_lineage = spark.read.table(f"{CATALOG}.{SILVER_SCHEMA}.silver_datahub_lineage")

# Join dbt models with GitHub issues by matching model name against issue titles/bodies
df_gold_impact = df_silver_dbt.alias("dbt") \
    .join(
        df_silver_issues.alias("issues"),
        lower(col("issues.search_text")).contains(lower(col("dbt.model_name"))),
        "left"
    ) \
    .select(
        col("dbt.model_name"),
        col("dbt.resource_type"),
        col("dbt.database"),
        col("dbt.schema"),
        col("dbt.upstream_depends_on"),
        col("issues.issue_number").alias("linked_issue_number"),
        col("issues.title").alias("linked_issue_title"),
        col("issues.state").alias("linked_issue_state"),
        col("issues.html_url").alias("linked_issue_url")
    ) \
    .withColumn(
        "has_active_incident",
        when((col("linked_issue_state") == "open"), lit(True)).otherwise(lit(False))
    ) \
    .withColumn(
        "risk_level",
        when(col("has_active_incident") == True, lit("HIGH"))
        .when(size(col("upstream_depends_on")) > 0, lit("MEDIUM"))
        .otherwise(lit("LOW"))
    ) \
    .withColumn("created_at", current_timestamp())

save_to_gold(df_gold_impact, "gold_pipeline_impact_risk")


# ---------------------------------------------------------
# 2. GOLD: Unified Entity Knowledge Graph
# ---------------------------------------------------------
print("\n--- [2/2] Building Gold: Entity Knowledge Graph ---")

# Standardize nodes across dbt models and chunked documentation
df_dbt_nodes = df_silver_dbt.select(
    concat_ws("::", lit("dbt"), col("model_name")).alias("entity_urn"),
    lit("dbt_model").alias("entity_type"),
    col("model_name").alias("entity_name"),
    col("description").alias("entity_description")
)

df_doc_nodes = spark.read.table(f"{CATALOG}.{SILVER_SCHEMA}.silver_documentation_chunked") \
    .select(
        concat_ws("::", lit("doc"), col("doc_id")).alias("entity_urn"),
        lit("documentation").alias("entity_type"),
        col("title").alias("entity_name"),
        col("chunk_text").alias("entity_description")
    )

# Union the datasets using allowMissingColumns=True
df_gold_knowledge_graph = df_dbt_nodes.unionByName(df_doc_nodes, allowMissingColumns=True) \
    .withColumn("processed_at", current_timestamp())

save_to_gold(df_gold_knowledge_graph, "gold_entity_knowledge_graph")

print("\n🎉 GOLD LAYER COMPLETE! PIPELINE INTELLIGENCE LAKEHOUSE FULLY BUILT.")